# Chang Zhou: occupation and task cleaning

This notebook runs the accepted rules from `plan.md` in memory using `clean_data.py`. It reads the five raw CSVs, checks hashes, and displays validation evidence. It does not overwrite source files or write outputs. Run all cells in a fresh kernel from the repository root or this folder.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

working = Path.cwd()
script_dir = working if (working / 'clean_data.py').is_file() else working / 'code' / 'cleaning_chang'
assert (script_dir / 'clean_data.py').is_file(), 'Start Jupyter from the repository root or code/cleaning_chang'
sys.path.insert(0, str(script_dir))
from clean_data import ROOT, load_sources, clean_sources, build_outputs, record_checks, cleaned_sample
print('Repository:', ROOT)

Repository: /Users/kristta/dso576/DSO576-Project-


## Source hashes and source-grain transformations

Raw values remain as strings in the source-grain copies. New SOC keys, normalized task keys, and numeric fields are derived separately. The script stops if any hash or schema differs from the accepted plan.

In [2]:
raw, source_hashes = load_sources(ROOT / 'data')
report = {'stages': [], 'source_version': 'unknown', 'download_date': 'unknown'}
clean = clean_sources(raw, report)
display(pd.DataFrame([{'file': name, 'sha256': digest} for name, digest in source_hashes.items()]))
display(pd.DataFrame([{'stage': item['stage'], 'rows': item['profile']['rows'], 'missing_cells': sum(item['profile']['missing_by_column'].values()), 'duplicate_rows': item['profile']['exact_duplicate_rows_excluding_source_row'], 'note': item['note']} for item in report['stages']]))

raw_job_exposure: rows=756; missing=0; duplicate_rows=0; read without changing source values
raw_task_penetration: rows=17,998; missing=0; duplicate_rows=6; read without changing source values
raw_onet_task_statements: rows=18,838; missing=1005; duplicate_rows=0; read without changing source values
raw_onet_occupation_data: rows=1,016; missing=0; duplicate_rows=0; read without changing source values
raw_bls: rows=1,112; missing=1037; duplicate_rows=0; read without changing source values
  categories: {"Occupation type": {"Line item": 831, "Summary": 281}}
clean_job_exposure: rows=756; missing=0; duplicate_rows=0; added six-digit key and numeric exposure; zero source rows removed
  numeric: {"observed_exposure_value": {"nonmissing": 756, "min": 0.0, "max": 0.7451, "negative": 0, "zero": 411}}
clean_task_penetration: rows=17,998; missing=0; duplicate_rows=6; source rows retained; 7 repeated normalized-key rows flagged
  numeric: {"penetration_value": {"nonmissing": 17998, "min": 0.0, "ma

,file,sha256
0,job_exposure.csv,4f0a3adf5feeb2ec5f5d02ab18cc5e851a2a4b8470bde8...
1,task_penetration.csv,85bee872db1d55d3e9a7f4e89da5ae4a5d59aa8ec875d7...
2,onet_task_statements.csv,4568e14d8db6f05551ad37e056e909b13ebae4a26bcb5f...
3,onet_occupation_data.csv,a09eae1d6609686e44e05b7290993a1c8b523d8ca224bc...
4,bls_table_1_2_2025_35.csv,460af8e1eaf961732bc29b56cc92b25c4d6ef85c587b8b...


,stage,rows,missing_cells,duplicate_rows,note
0,raw_job_exposure,756,0,0,read without changing source values
1,raw_task_penetration,17998,0,6,read without changing source values
2,raw_onet_task_statements,18838,1005,0,read without changing source values
3,raw_onet_occupation_data,1016,0,0,read without changing source values
4,raw_bls,1112,1037,0,read without changing source values
5,clean_job_exposure,756,0,0,added six-digit key and numeric exposure; zero...
6,clean_task_penetration,17998,0,6,source rows retained; 7 repeated normalized-ke...
7,clean_onet_occupations,1016,0,0,all child occupation records preserved
8,clean_onet_tasks,18838,2010,0,all .00 and child task statements preserved
9,clean_bls,1112,3146,0,raw units retained; missing symbols mapped onl...


## Occupation and occupation-task joins

BLS Summary rows stay separate. The exposure-left occupation output retains unmatched occupations. All O*NET task statements, including child O*NET-SOC codes, contribute to task matching; repeated normalized tasks within the same six-digit SOC are counted once in the derived occupation-task table.

In [3]:
outputs = build_outputs(clean, report)
display(pd.DataFrame([{'stage': item['stage'], 'rows': item['profile']['rows'], 'missing_cells': sum(item['profile']['missing_by_column'].values()), 'duplicate_rows': item['profile']['exact_duplicate_rows_excluding_source_row'], 'note': item['note']} for item in report['stages']]))
display(pd.DataFrame(report['bls_line_item_totals']).T)
display(outputs['occupation'][['soc_code', 'title', 'observed_exposure', 'bls_match_status', 'employment_2025_thousands', 'average_task_penetration', 'matched_task_count']].head())

bls_line_items: rows=831; missing=2277; duplicate_rows=0; separate grain; 281 Summary rows retained separately
  numeric: {"employment_2025_thousands": {"nonmissing": 831, "min": 0.2, "max": 4677.1, "negative": 0, "zero": 0}, "employment_change_percent_2025_35": {"nonmissing": 831, "min": -34.4, "max": 41.0, "negative": 232, "zero": 8}, "annual_openings_2025_35_thousands": {"nonmissing": 831, "min": 0.0, "max": 827.4, "negative": 0, "zero": 3}, "median_annual_wage_2025_dollars": {"nonmissing": 825, "min": 30890.0, "max": 559030.0, "negative": 0, "zero": 0}}
  categories: {"typical_education": {"High school diploma or equivalent": 326, "Bachelor's degree": 177, "No formal educational credential": 109, "Doctoral or professional degree": 73, "Postsecondary nondegree award": 51, "Associate's degree": 48, "Master's degree": 40, "Some college, no degree": 7}, "related_work_experience": {"<missing>": 716, "Less than 5 years": 87, "5 years or more": 28}, "on_the_job_training": {"<missing>": 32

,stage,rows,missing_cells,duplicate_rows,note
0,raw_job_exposure,756,0,0,read without changing source values
1,raw_task_penetration,17998,0,6,read without changing source values
2,raw_onet_task_statements,18838,1005,0,read without changing source values
3,raw_onet_occupation_data,1016,0,0,read without changing source values
4,raw_bls,1112,1037,0,read without changing source values
5,clean_job_exposure,756,0,0,added six-digit key and numeric exposure; zero...
6,clean_task_penetration,17998,0,6,source rows retained; 7 repeated normalized-ke...
7,clean_onet_occupations,1016,0,0,all child occupation records preserved
8,clean_onet_tasks,18838,2010,0,all .00 and child task statements preserved
9,clean_bls,1112,3146,0,raw units retained; missing symbols mapped onl...


,unit,all_line_items,matched_to_exposure,unmatched_to_exposure,occupation_output_matched_sum,sum_difference
employment_2025_thousands,thousands,"{'rows': 831, 'nonmissing': 831, 'sum': 170267.2}","{'rows': 755, 'nonmissing': 755, 'sum': 154223.4}","{'rows': 76, 'nonmissing': 76, 'sum': 16043.80...",154223.4,0.0
annual_openings_2025_35_thousands,thousands per year,"{'rows': 831, 'nonmissing': 831, 'sum': 17460.7}","{'rows': 755, 'nonmissing': 755, 'sum': 15607.2}","{'rows': 76, 'nonmissing': 76, 'sum': 1853.499...",15607.2,0.0


,soc_code,title,observed_exposure,bls_match_status,employment_2025_thousands,average_task_penetration,matched_task_count
0,11-1011,Chief Executives,0.0333,matched,291.6,0.032945,49
1,11-1021,General and Operations Managers,0.1378,matched,3599.0,0.102653,17
2,11-1031,Legislators,0.0,unmatched,<NA>,0.0,30
3,11-2011,Advertising and Promotions Managers,0.1731,matched,33.2,0.171673,30
4,11-2021,Marketing Managers,0.3195,matched,421.6,0.194865,20


## Missing/sentinel conversion and record checks

Each affected source column is reconciled to the missing count in its derived column. The five record checks show a specific source file and row/key, original value, prewritten expected result, actual result, and pass/fail.

In [4]:
for source_name in ['onet_missing_conversion', 'bls_missing_conversion']:
    print(source_name)
    display(pd.DataFrame.from_dict(report[source_name], orient='index'))
checks = record_checks(outputs, clean)
display(checks[['case', 'source_file', 'source_row', 'source_key', 'original_value', 'expected_value', 'actual_value', 'pass']])
assert checks['pass'].all()
print('Cleaned sample rows:', len(cleaned_sample(outputs['occupation'])))

onet_missing_conversion


,source_blank,source_em_dash,source_missing_or_sentinel,derived_missing,derived_nonmissing
Task Type,418,0,418,418,18420
Incumbents Responding,587,0,587,587,18251


bls_missing_conversion


,source_blank,source_em_dash,source_missing_or_sentinel,derived_missing,derived_nonmissing
"Employment, 2025",0,0,0,0,1112
"Employment, 2035",0,0,0,0,1112
"Employment distribution, percent, 2025",0,0,0,0,1112
"Employment distribution, percent, 2035",0,0,0,0,1112
"Employment change, numeric, 2025–35",0,0,0,0,1112
"Employment change, percent, 2025–35",0,0,0,0,1112
"Percent self-employed, 2025",0,220,220,220,892
"Occupational openings, 2025–35 annual average",0,0,0,0,1112
"Median annual wage, dollars, 2025",0,9,9,9,1103
Typical education needed for entry,0,281,281,281,831


,case,source_file,source_row,source_key,original_value,expected_value,actual_value,pass
0,Chief Executives,job_exposure.csv,2,occ_code=11-1011,"{""observed_exposure"": ""0.0333"", ""title"": ""Chie...","{""output_rows"": 1, ""exposure_text"": ""0.0333"", ...","{""output_rows"": 1, ""exposure_text"": ""0.0333"", ...",True
1,Legislators,job_exposure.csv,4,occ_code=11-1031,"{""observed_exposure"": ""0.0"", ""title"": ""Legisla...","{""output_rows"": 1, ""exposure_text"": ""0.0"", ""bl...","{""output_rows"": 1, ""exposure_text"": ""0.0"", ""bl...",True
2,Top executives Summary,bls_table_1_2_2025_35.csv,4,2025 National Employment Matrix code=11-1000,"{""title"": "" Top executives"", ""occupation_ty...","{""summary_rows"": 1, ""occupation_rows"": 0, ""occ...","{""summary_rows"": 1, ""occupation_rows"": 0, ""occ...",True
3,Press-release duplicate task,task_penetration.csv,17891,task_normalized=write interesting and effectiv...,"{""task"": ""Write interesting and effective pres...","{""source_rows"": 8, ""lookup_rows"": 1, ""normaliz...","{""source_rows"": 8, ""lookup_rows"": 1, ""normaliz...",True
4,Student-organization advisers,onet_task_statements.csv,5886,O*NET-SOC Code=25-1011.00; Task ID=5682,"{""Task"": ""Act as advisers to student organizat...","{""soc_code"": ""25-1011"", ""task_text"": ""Act as a...","{""soc_code"": ""25-1011"", ""task_text"": ""Act as a...",True


Cleaned sample rows: 26
